5. Model Training

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib

DATA, MODELS, METRICS = ROOT/'data/processed/final_dataset.csv', ROOT/'models', ROOT/'results/metrics'
MODELS.mkdir(parents=True, exist_ok=True); METRICS.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
y = df.pop('overdue_label')

# Explicitly exclude IDs, free text, and fields unavailable at raise time.
drop = ['description','created','target','priority','cause','documents','comments','images','association','to_package']
X = df.drop(columns=[c for c in drop if c in df], errors='ignore')
cat = X.select_dtypes(include=['object']).columns.tolist(); num = [c for c in X.columns if c not in cat]

pre = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num), 
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)

# calculating the imbalanced ration for XGBoost Model
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

# defining models
models = {
    'logistic_regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'random_forest': RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1),
    'xgboost': XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, scale_pos_weight=ratio, random_state=42, n_jobs=-1),
    'lightgbm': LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1),
    'catboost': CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, auto_class_weights='Balanced', random_state=42, verbose=0)
}

cv = StratifiedKFold(5, shuffle=True, random_state=42); rows=[]
for name, model in models.items():
    pipe = Pipeline([('preprocessor', pre), ('model', model)])
    score = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['roc_auc','precision','recall','f1','accuracy'], n_jobs=-1)
    rows.append({'model':name, **{k.replace('test_','cv_'):v.mean() for k,v in score.items() if k.startswith('test_')}})

metrics = pd.DataFrame(rows).sort_values('cv_f1', ascending=False); metrics.to_csv(METRICS/'cross_validation_metrics.csv', index=False)
best_name = metrics.iloc[0]['model']; best = Pipeline([('preprocessor', pre), ('model', models[best_name])]); best.fit(X_train,y_train)
joblib.dump(best, MODELS/'overdue_risk_model.joblib'); joblib.dump({'X_test':X_test,'y_test':y_test,'feature_columns':X.columns.tolist()}, MODELS/'evaluation_holdout.joblib')
metrics

,model,cv_roc_auc,cv_precision,cv_recall,cv_f1,cv_accuracy
3,lightgbm,0.983806,0.521291,0.935484,0.669394,0.936352
2,xgboost,0.983075,0.479280,0.940029,0.634350,0.925211
4,catboost,0.982318,0.466621,0.941533,0.623670,0.921601
0,logistic_regression,0.980796,0.441745,0.937055,0.600300,0.914070
1,random_forest,0.976627,0.335900,0.991000,0.501632,0.864349
